# Simple Graph-Powered RAG with OraclePageIndex

<p align="center"><b>Oracle SQL Property Graphs &middot; No Vector DB &middot; No Embeddings &middot; Graph-Based Reasoning</b></p>

---

## Introduction

**OraclePageIndex** is a reasoning-based, vectorless RAG framework that performs retrieval in two steps:

1. **Parse a PDF** into a hierarchical tree structure using an LLM (Ollama)
2. **Search the tree** with LLM reasoning to find relevant sections — no vector similarity needed

This is the same approach as [PageIndex](https://github.com/VectifyAI/PageIndex), but running entirely locally:
- **Ollama** replaces OpenAI for all LLM calls
- **Oracle Property Graphs** replace JSON files for persistent storage (shown in other notebooks)
- **SQL/PGQ** replaces ad-hoc search with graph traversal queries

## Notebook Overview

This notebook demonstrates a minimal, local example of **graph-powered RAG**:

- [x] Build a document tree structure locally with Ollama
- [x] Perform reasoning-based retrieval via LLM tree search
- [x] Generate answers from retrieved context

> **Prerequisites:** [Ollama](https://ollama.com/) running locally with a model pulled (e.g., `ollama pull mistral:7b`)

---

## Step 0: Preparation

### 0.1 Setup Path

In [1]:
import sys, os

# Add project root to path so we can import oracle_pageindex
sys.path.insert(0, os.path.abspath(".."))

# Alternatively: pip install -e .. (from project root)

### 0.2 Configure Ollama

In [2]:
from oracle_pageindex.llm import OllamaClient
from oracle_pageindex.parser import DocumentParser

# Ollama must be running locally with a model pulled
# e.g.: ollama pull mistral:7b
OLLAMA_MODEL = "mistral:7b"  # Change to your preferred model

llm = OllamaClient(
    base_url="http://localhost:11434",
    model=OLLAMA_MODEL,
    temperature=0,
    num_ctx=8192,
)

# Quick test
response = llm.chat("Say 'hello' in one word.")
print(f"Ollama ({OLLAMA_MODEL}) ready: {response}")

Ollama (mistral:7b) ready:  Hello!


## Step 1: Parse PDF and Build Tree

OraclePageIndex parses PDFs into a hierarchical tree. The LLM extracts the document's
table of contents / section structure, then generates a summary for each section.

### 1.1 Download and parse a document

In [3]:
from urllib.request import urlretrieve

# Download "Attention Is All You Need" (15 pages)
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"
pdf_path = os.path.join("data", "attention-is-all-you-need.pdf")
os.makedirs("data", exist_ok=True)

if not os.path.exists(pdf_path):
    print(f"Downloading {pdf_url}...")
    urlretrieve(pdf_url, pdf_path)
    print(f"Saved to {pdf_path}")
else:
    print(f"Using cached {pdf_path}")

# Build the document tree using Ollama
parser = DocumentParser(
    llm=llm,
    toc_check_page_num=15,
    max_token_num_each_node=20000,
    add_node_id=True,
    add_summaries=True,
)

print("\nBuilding document tree with Ollama (this takes a few minutes)...")
result = parser.build_tree(pdf_path)

doc_name = result["doc_name"]
tree = result["structure"]
page_list = result["page_list"]

print(f"\nDocument: {doc_name}")
print(f"Pages: {len(page_list)}")

Using cached data/attention-is-all-you-need.pdf

Building document tree with Ollama (this takes a few minutes)...


Failed to parse JSON from LLM response


Failed to parse JSON from response


Could not parse ToC from LLM response


ToC extraction failed — using fallback structure



Document: attention-is-all-you-need.pdf
Pages: 15


### 1.2 Visualize the tree structure

In [4]:
import copy, json, textwrap


def print_tree(node, indent=0):
    """Print tree structure in a readable format."""
    if isinstance(node, list):
        for item in node:
            print_tree(item, indent)
        return
    title = node.get("title", "Untitled")
    node_id = node.get("node_id", "?")
    summary = node.get("summary", "")
    if summary and len(summary) > 60:
        summary = summary[:60] + "..."
    start = node.get("start_index", "?")
    end = node.get("end_index", "?")
    prefix = "  " * indent
    print(f"{prefix}[{node_id}] {title} (pages {start}-{end})")
    if summary:
        print(f"{prefix}  Summary: {summary}")
    for child in node.get("nodes", []):
        print_tree(child, indent + 1)


def create_node_map(node, result=None):
    """Flatten tree into node_id -> node dict."""
    if result is None:
        result = {}
    if isinstance(node, list):
        for item in node:
            create_node_map(item, result)
        return result
    if isinstance(node, dict):
        nid = node.get("node_id")
        if nid:
            result[nid] = node
        for child in node.get("nodes", []):
            create_node_map(child, result)
    return result


def strip_text(node):
    """Return tree copy without 'text' fields (for compact LLM prompts)."""
    if isinstance(node, list):
        return [strip_text(item) for item in node]
    if isinstance(node, dict):
        return {
            k: (strip_text(v) if k == "nodes" else v)
            for k, v in node.items()
            if k != "text"
        }
    return node


print("Document Tree Structure:\n")
print_tree(tree)

Document Tree Structure:

[0000] Page 1 (pages 1-2)
  Summary:  The section discusses a paper titled "Attention Is All You ...
[0001] Page 2 (pages 2-3)
  Summary:  This section discusses the Transformer model architecture, ...
[0002] Page 3 (pages 3-4)
  Summary:  The section discusses the architecture of the Transformer m...
[0003] Page 4 (pages 4-5)
  Summary:  The section discusses Scaled Dot-Product Attention and Mult...
[0004] Page 5 (pages 5-6)
  Summary:  This section discusses the main components of the Transform...
[0005] Page 6 (pages 6-7)
  Summary:  This section discusses the use of self-attention layers in ...
[0006] Page 7 (pages 7-8)
  Summary:  This section discusses several key points related to machin...
[0007] Page 8 (pages 8-9)
  Summary:  This section discusses the results of a study comparing the...
[0008] Page 9 (pages 9-10)
  Summary:  This section discusses experiments on variations of the Tra...
[0009] Page 10 (pages 10-11)
  Summary:  This section presents r

## Step 2: Reasoning-Based Retrieval with Tree Search

Instead of embedding chunks and doing cosine similarity, we send the tree structure to the
LLM and ask it to **reason** about which nodes contain the answer — just like a human
scanning a table of contents.

### 2.1 Search the tree with Ollama

In [5]:
query = "What is the BLEU score achieved by the Transformer on the English-to-German translation task?"

# Prepare the tree for the LLM (remove raw text to save tokens)
tree_for_llm = strip_text(tree)

search_prompt = (
    "You are given a question and a tree structure of a document.\n"
    "Each node has a node_id, title, and summary.\n"
    "Find all nodes that are likely to contain the answer to the question.\n\n"
    f"Question: {query}\n\n"
    f"Document tree:\n{json.dumps(tree_for_llm, indent=2)}\n\n"
    'Reply in this exact JSON format:\n'
    '{"thinking": "<your reasoning about which nodes are relevant>", '
    '"node_list": ["node_id_1", "node_id_2"]}\n'
    "Return ONLY valid JSON. No other text."
)

print("Searching tree with Ollama...\n")
search_result = llm.chat(search_prompt)
print(search_result)

Searching tree with Ollama...



 {"thinking": "The question asks for the BLEU score achieved by the Transformer on the English-to-German translation task. The document discusses the Transformer model and its performance in detail, including new single-model state-of-the-art BLEU scores on the WMT 2014 English-to-German and English-to-French translation tasks. Therefore, the relevant nodes are those that discuss the results of the Transformer on the English-to-German translation task.", "node_list": ["0008", "0009"]}


### 2.2 View retrieved nodes and reasoning

In [6]:
node_map = create_node_map(tree)
result_json = llm.extract_json(search_result)

print("Reasoning:\n")
print(textwrap.fill(result_json.get("thinking", ""), width=90))

print("\nRetrieved Nodes:\n")
for nid in result_json.get("node_list", []):
    node = node_map.get(nid, {})
    title = node.get("title", "?")
    start = node.get("start_index", "?")
    end = node.get("end_index", "?")
    print(f"  [{nid}] {title} (pages {start}-{end})")

Reasoning:

The question asks for the BLEU score achieved by the Transformer on the English-to-German
translation task. The document discusses the Transformer model and its performance in
detail, including new single-model state-of-the-art BLEU scores on the WMT 2014 English-
to-German and English-to-French translation tasks. Therefore, the relevant nodes are those
that discuss the results of the Transformer on the English-to-German translation task.

Retrieved Nodes:

  [0008] Page 9 (pages 9-10)
  [0009] Page 10 (pages 10-11)


## Step 3: Answer Generation

### 3.1 Extract context from retrieved nodes

In [7]:
node_list = result_json.get("node_list", [])
context_parts = []

for nid in node_list:
    node = node_map.get(nid, {})
    text = node.get("text", "")
    title = node.get("title", "Untitled")
    if text:
        context_parts.append(f"## {title}\n\n{text}")

relevant_context = "\n\n---\n\n".join(context_parts)

print("Retrieved Context (first 1000 chars):\n")
print(textwrap.fill(relevant_context[:1000], width=90))
print("...")

Retrieved Context (first 1000 chars):

## Page 9  Table 3: Variations on the Transformer architecture. Unlisted values are
identical to those of the base model. All metrics are on the English-to-German translation
development set, newstest2013. Listed perplexities are per-wordpiece, according to our
byte-pair encoding, and should not be compared to per-word perplexities. N dmodel dff h dk
dv Pdrop ϵls train PPL BLEU params steps (dev) (dev) ×106 base 6 512 2048 8 64 64 0.1 0.1
100K 4.92 25.8 65 (A) 1 512 512 5.29 24.9 4 128 128 5.00 25.5 16 32 32 4.91 25.8 32 16 16
5.01 25.4 (B) 16 5.16 25.1 58 32 5.01 25.4 60 (C) 2 6.11 23.7 36 4 5.19 25.3 50 8 4.88
25.5 80 256 32 32 5.75 24.5 28 1024 128 128 4.66 26.0 168 1024 5.12 25.4 53 4096 4.75 26.2
90 (D) 0.0 5.77 24.6 0.2 4.95 25.5 0.0 4.67 25.3 0.2 5.47 25.7 (E) positional embedding
instead of sinusoids 4.92 25.7 big 6 1024 4096 16 0.3 300K 4.33 26.4 213 development set,
newstest2013. We used beam search as described in the previous section, 

### 3.2 Generate answer with Ollama

In [8]:
answer_prompt = (
    "Answer the question based on the provided document context.\n\n"
    f"Question: {query}\n\n"
    f"Context:\n{relevant_context}\n\n"
    "Provide a clear, concise answer based only on the context provided."
)

print("Generating answer with Ollama...\n")
answer = llm.chat(answer_prompt)

print("Answer:\n")
print(textwrap.fill(answer, width=90))

Generating answer with Ollama...



Answer:

 The BLEU score achieved by the Transformer on the English-to-German translation task is
not explicitly stated in the provided document context. However, it is mentioned that for
this task, the best model outperforms all previously reported ensembles, suggesting a high
BLEU score.


---

## What's Next

This notebook showed the core concept: **reasoning-based retrieval over a document tree**,
running entirely locally with Ollama.

The other cookbook notebooks build on this foundation:

- **[Oracle Graph Quickstart](oracle_graph_quickstart.ipynb)** — Store the tree in an Oracle Property Graph and query with SQL/PGQ
- **[Graph Retrieval](graph_retrieval.ipynb)** — Deep dive into SQL/PGQ structured retrieval with entity search and multi-hop traversal
- **[Vision RAG](vision_RAG_oracle.ipynb)** — Vision-based RAG using multimodal Ollama (no OCR needed)

---

*Built with [OraclePageIndex](https://github.com/jasperan/OraclePageIndex) — Oracle AI Database powered document intelligence with Property Graphs.*